# 🔧 Advanced Feature Engineering for Ethiopian Salary Prediction

## Overview
This notebook focuses on advanced feature engineering techniques to enhance our salary prediction model. We'll create new features, transform existing ones, and prepare the data for optimal machine learning performance.

### Key Techniques:
- 🎯 **Feature Creation**: Derive meaningful features from existing data
- 🔄 **Feature Transformation**: Apply mathematical transformations
- 📊 **Encoding Techniques**: Handle categorical variables effectively
- ⚖️ **Feature Scaling**: Normalize features for better model performance
- 🎨 **Polynomial Features**: Create interaction terms
- 📈 **Feature Selection**: Identify most important features

---

In [ ]:
# Import comprehensive libraries for feature engineering
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn for feature engineering
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, OrdinalEncoder,
    PolynomialFeatures, PowerTransformer
)
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, SelectFromModel
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Advanced feature engineering libraries
from feature_engine.creation import MathematicalCombination
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.encoding import RareLabelEncoder

print("🔧 Feature Engineering Libraries Loaded Successfully!")
print("📊 Ready for Advanced Feature Engineering")

In [ ]:
# Load and prepare the dataset
print("📥 Loading Ethiopian Salary Dataset for Feature Engineering")
print("=" * 60)

# Load data
df = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# Create a copy for feature engineering
df_fe = df.copy()

# Basic preprocessing
df_fe['salary_usd'] = df_fe['salary_etb'] / 55  # ETB to USD conversion

# Handle missing values in test_score
df_fe['test_score'].fillna(df_fe['test_score'].median(), inplace=True)
print(f"✅ Missing values handled: {df_fe.isnull().sum().sum()} remaining")

print("\n🎯 Starting Advanced Feature Engineering Process...")
print("-" * 50)

## 🎨 Creating New Features

Let's create meaningful features that can improve our model's predictive power.

In [ ]:
# 1. EXPERIENCE-BASED FEATURES
print("👨‍💼 Creating Experience-Based Features")
print("-" * 40)

# Experience categories
df_fe['experience_level'] = pd.cut(
    df_fe['experience_years'], 
    bins=[0, 1, 3, 5, 8, float('inf')], 
    labels=['Entry', 'Junior', 'Mid', 'Senior', 'Expert']
)

# Experience squared (non-linear relationship)
df_fe['experience_squared'] = df_fe['experience_years'] ** 2

# Experience log transformation
df_fe['experience_log'] = np.log1p(df_fe['experience_years'])

# Experience bins
df_fe['experience_bin'] = pd.cut(df_fe['experience_years'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

print(f"✅ Created 4 experience-based features")

# 2. TEST SCORE FEATURES
print("\n📝 Creating Test Score Features")
print("-" * 40)

# Test score categories
df_fe['test_score_category'] = pd.cut(
    df_fe['test_score'], 
    bins=[0, 70, 80, 90, 100], 
    labels=['Below Average', 'Average', 'Good', 'Excellent']
)

# Test score normalized (0-1 scale)
df_fe['test_score_normalized'] = df_fe['test_score'] / 100

# Test score squared
df_fe['test_score_squared'] = df_fe['test_score'] ** 2

print(f"✅ Created 3 test score features")

# 3. INTERACTION FEATURES
print("\n🔗 Creating Interaction Features")
print("-" * 40)

# Experience × Test Score interaction
df_fe['experience_test_interaction'] = df_fe['experience_years'] * df_fe['test_score']

# Experience to Test Score ratio
df_fe['experience_test_ratio'] = df_fe['experience_years'] / (df_fe['test_score'] + 1)  # +1 to avoid division by zero

# Combined skill score (weighted combination)
df_fe['combined_skill_score'] = (df_fe['experience_years'] * 0.6) + (df_fe['test_score'] * 0.4)

print(f"✅ Created 3 interaction features")

# 4. LOCATION-BASED FEATURES
print("\n🌍 Creating Location-Based Features")
print("-" * 40)

# Capital city indicator
df_fe['is_addis_ababa'] = (df_fe['location'] == 'Addis Ababa').astype(int)

# Major city indicator (top 5 cities by frequency)
top_cities = df_fe['location'].value_counts().head(5).index
df_fe['is_major_city'] = df_fe['location'].isin(top_cities).astype(int)

# Location salary rank (based on average salary)
location_salary_rank = df_fe.groupby('location')['salary_etb'].mean().rank(ascending=False)
df_fe['location_salary_rank'] = df_fe['location'].map(location_salary_rank)

print(f"✅ Created 3 location-based features")

# 5. DEPARTMENT-BASED FEATURES
print("\n🏢 Creating Department-Based Features")
print("-" * 40)

# High-paying department indicator
dept_avg_salary = df_fe.groupby('department')['salary_etb'].mean()
high_paying_depts = dept_avg_salary[dept_avg_salary > dept_avg_salary.median()].index
df_fe['is_high_paying_dept'] = df_fe['department'].isin(high_paying_depts).astype(int)

# Department salary rank
dept_salary_rank = dept_avg_salary.rank(ascending=False)
df_fe['dept_salary_rank'] = df_fe['department'].map(dept_salary_rank)

print(f"✅ Created 2 department-based features")

print(f"\n🎉 Total new features created: {df_fe.shape[1] - df.shape[1]}")
print(f"📊 Dataset now has {df_fe.shape[1]} columns")